<a href="https://colab.research.google.com/github/yanaguntikarmeesal/Agentic-AI-Arithmetic-Assistant-Learning-Project/blob/main/Agentic_AI_Arithmetic_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Agentic AI Arithmetic Assistant Using Groq + LangGraph**

# Step 1 — Install the required libraries

In [ ]:
# ============================================================
# CELL 1: INSTALL LIBRARIES
# ============================================================

%pip install -q -U langchain langchain-groq langgraph langchain-core typing_extensions

# Step 2 — Configure your Groq API key

Create an API key from the Groq Console
.

In [ ]:
# ============================================================
# CELL 2: API KEY CONFIGURATION
# ============================================================

import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass(
        "Enter your Groq API key: "
    )

print("API key configured successfully.")

# Step 3 — Import libraries and initialize the model

We use openai/gpt-oss-20b rather than the unavailable model from your previous notebook.

In [ ]:
# ============================================================
# CELL 3: IMPORT LIBRARIES AND INITIALIZE MODEL
# ============================================================

from langchain_groq import ChatGroq
from langchain.tools import tool

from langchain_core.messages import (
    AnyMessage,
    SystemMessage,
    HumanMessage,
    ToolMessage,
)

from typing_extensions import TypedDict, Annotated
from typing import Literal
import operator

from langgraph.graph import StateGraph, START, END

# Initialize the Groq model
MODEL_NAME = "openai/gpt-oss-20b"

model = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
)

print("Model initialized:", MODEL_NAME)

# Step 4 — Define arithmetic tools

The @tool decorator makes each Python function available to the language model.

In [ ]:
# ============================================================
# CELL 4: DEFINE ARITHMETIC TOOLS
# ============================================================

@tool
def add(a: int, b: int) -> int:
    """Add two integers.

    Args:
        a: First integer.
        b: Second integer.
    """
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers.

    Args:
        a: First integer.
        b: Second integer.
    """
    return a * b


@tool
def divide(a: int, b: int) -> float:
    """Divide the first integer by the second integer.

    Args:
        a: Numerator.
        b: Denominator.
    """
    if b == 0:
        raise ValueError("Cannot divide by zero.")

    return a / b


# Register tools
tools = [add, multiply, divide]

# Dictionary for tool lookup
tools_by_name = {
    item.name: item for item in tools
}

# Bind tools to the model
model_with_tools = model.bind_tools(tools)

print("Registered tools:")
for item in tools:
    print("-", item.name)

# Step 5 — Define the agent state

The state stores messages and counts how many times the model is called.

operator.add ensures that new messages are appended to the existing list.

In [ ]:
# ============================================================
# CELL 5: DEFINE AGENT STATE
# ============================================================

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int


print("Agent state defined successfully.")

# Step 6 — Define the LLM node

The LLM node sends the system instructions and conversation messages to the model.

Important: The correct message combination is [system_message] + state["messages"].

In [ ]:
# ============================================================
# CELL 6: DEFINE LLM NODE
# ============================================================

def llm_call(state: MessagesState):
    """Call the model and decide whether to use a tool."""

    system_message = SystemMessage(
        content=(
            "You are a helpful arithmetic assistant. "
            "Use the available tools for addition, "
            "multiplication, and division. "
            "Use tools to perform calculations. "
            "If a calculation involves multiple steps, "
            "perform them in the correct order. "
            "Explain the final answer clearly."
        )
    )

    # Correctly combine system and conversation messages
    conversation = [system_message] + state["messages"]

    # Invoke the model
    response = model_with_tools.invoke(conversation)

    return {
        "messages": [response],
        "llm_calls": state.get("llm_calls", 0) + 1,
    }


print("LLM node created successfully.")

# Step 7 — Define the tool execution node

This node executes each tool requested by the model and returns its result.

In [ ]:
# ============================================================
# CELL 7: DEFINE TOOL NODE
# ============================================================

def tool_node(state: MessagesState):
    """Execute tools requested by the model."""

    result = []

    last_message = state["messages"][-1]

    for tool_call in last_message.tool_calls:

        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        selected_tool = tools_by_name.get(tool_name)

        if selected_tool is None:
            observation = f"Unknown tool: {tool_name}"

        else:
            try:
                observation = selected_tool.invoke(tool_args)
                observation = str(observation)

            except Exception as error:
                observation = f"Tool error: {error}"

        result.append(
            ToolMessage(
                content=observation,
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": result}


print("Tool node created successfully.")

# Step 8 — Define conditional routing

The agent checks whether the model requested a tool.

1. If there are tool calls, it moves to tool_node.

2. If there are no tool calls, the workflow ends.

In [ ]:
# ============================================================
# CELL 8: DEFINE CONDITIONAL ROUTING
# ============================================================

def should_continue(
    state: MessagesState,
) -> Literal["tool_node", "__end__"]:

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tool_node"

    return END


print("Conditional routing created successfully.")

# Step 9 — Build and compile the LangGraph agent

In [ ]:
# ============================================================
# CELL 9: BUILD AND COMPILE AGENT
# ============================================================

agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Start workflow
agent_builder.add_edge(START, "llm_call")

# Conditional routing
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END],
)

# Return tool results to the model
agent_builder.add_edge("tool_node", "llm_call")

# Compile the graph
agent = agent_builder.compile()

print("Agent compiled successfully.")

# Step 10 — Visualize the workflow

In [ ]:
# ============================================================
# CELL 10: DISPLAY LANGGRAPH WORKFLOW
# ============================================================

from IPython.display import Image, display

try:
    display(
        Image(
            agent.get_graph(xray=True).draw_mermaid_png()
        )
    )

except Exception as error:
    print("Could not render graph image:", error)
    print(agent.get_graph().draw_mermaid())

# Step 11 — Create a reusable function

This function lets you send different questions to the same agent.

In [ ]:
# ============================================================
# CELL 11: REUSABLE AGENT FUNCTION
# ============================================================

def run_agent(user_query: str):
    """Run the arithmetic agent with a user query."""

    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content=user_query)
            ],
            "llm_calls": 0,
        },
        config={
            "recursion_limit": 30
        },
    )

    return result


print("Agent function created successfully.")

# Step 12 — Test addition

In [ ]:
# ============================================================
# CELL 12: TEST ADDITION
# ============================================================

user_query = "Add 3 and 4."

response = run_agent(user_query)

print("USER:", user_query)
print("\nCONVERSATION:")

for message in response["messages"]:
    message.pretty_print()

print("\nTotal LLM calls:", response["llm_calls"])

# Step 13 — Test all three tools

In [ ]:
# ============================================================
# CELL 13: TEST MULTIPLE CALCULATIONS
# ============================================================

test_queries = [
    "Add 10 and 20.",
    "Multiply 6 and 7.",
    "Divide 100 by 5.",
    "Divide 10 by 3.",
    "Add 5 and 10, then multiply the result by 2.",
]

for query in test_queries:

    print("\n" + "=" * 60)
    print("USER:", query)

    try:
        result = run_agent(query)

        print(
            "ASSISTANT:",
            result["messages"][-1].content
        )

        print(
            "LLM calls:",
            result["llm_calls"]
        )

    except Exception as error:
        print("ERROR:", error)

# Step 14 — Test division by zero

In [ ]:
# ============================================================
# CELL 14: TEST DIVISION BY ZERO
# ============================================================

query = "Divide 10 by 0."

try:
    result = run_agent(query)

    print("USER:", query)
    print("ASSISTANT:", result["messages"][-1].content)

except Exception as error:
    print("Error:", error)

# Step 15 — Interactive arithmetic assistant

Enter your own calculations directly in Colab.

In [ ]:
# ============================================================
# CELL 15: INTERACTIVE ARITHMETIC ASSISTANT
# ============================================================

while True:

    user_input = input(
        "\nEnter a calculation (or type 'exit'): "
    ).strip()

    if user_input.lower() in ["exit", "quit"]:
        print("Calculator closed.")
        break

    if not user_input:
        print("Please enter a calculation.")
        continue

    try:
        result = run_agent(user_input)

        print("\n🤖 ASSISTANT:")
        print(result["messages"][-1].content)

        print("\nLLM calls:", result["llm_calls"])

    except Exception as error:
        print("\n❌ Error:", error)

# **Final project summary**

# Agentic AI Arithmetic Assistant

* Technologies

1. Python

2. Groq API — openai/gpt-oss-20b

3. LangChain

4. LangGraph


* Concepts learned

1. LLM initialization

2. Tool creation with @tool

3. State management with TypedDict

4. Model and tool nodes

5. Conditional edges

6. Graph compilation and invocation

7. Error handling and interactive testing

# Test the application

1. Add 10 and 20.

2. Multiply 6 and 7.

3. Divide 100 by 5.

4. Add 5 and 10, then multiply the result by 2.

2. Divide 10 by 0.